# FLUX.1-dev 4-bit transformer benchmark

This notebook is a **separate performance run**. It reuses the Hugging Face cache from the reference notebook, keeps the same prompt/seed/settings, and quantizes only the FLUX transformer to 4-bit NF4 using bitsandbytes. It does not alter the original reference notebook.

**Baseline:** 1024×1024 · 28 steps · CFG 3.5 · seed 123456789 · 3262.4 seconds · 5.89 GiB peak allocated VRAM on a Tesla T4.

Hugging Face documents 4-bit bitsandbytes quantization as a way to run FLUX.1-dev on GPUs with less than 16 GB VRAM. This may slightly change image quality; compare the result against the baseline.

The base model remains subject to its gated access and FLUX.1-dev Non-Commercial License. Use fictional adults only; no minors or non-consensual real people.

## 1. Install bitsandbytes

Run this cell, then use **Runtime → Restart session**. This removes the old TorchAO package that previously triggered the PEFT version check, while leaving the Hugging Face model cache on the runtime disk.

In [ ]:
%pip -q install -U bitsandbytes
%pip uninstall -y torchao


## 2. Verify the runtime and authenticate

After restarting, run these cells. You should again see the Tesla T4. If you already logged in, the token may still be cached; logging in again is fine.

In [ ]:
import torch, diffusers, bitsandbytes, platform
print('PyTorch:', torch.__version__)
print('Diffusers:', diffusers.__version__)
print('bitsandbytes:', bitsandbytes.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA runtime in Runtime → Change runtime type.')
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
if torch.cuda.get_device_capability(0) < (6, 0):
    raise RuntimeError('bitsandbytes 4-bit CUDA quantization requires compute capability 6.0 or newer.')


In [ ]:
from getpass import getpass
from huggingface_hub import login
hf_token = getpass('Hugging Face read token: ')
login(token=hf_token, add_to_git_credential=False)
del hf_token


## 3. Load the 4-bit transformer

This is the main change. Only the large FLUX transformer is quantized; CLIP, T5, and the VAE keep their normal FP16 weights. `device_map='auto'` places as much as possible on the GPU and offloads the remainder to CPU. Do **not** call `pipe4.to('cuda')` after loading with a device map.

The already-downloaded FP16 checkpoint should be reused from the Hugging Face cache. Quantization itself happens during load.

In [ ]:
import gc, time, torch
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig

BASE_MODEL = 'black-forest-labs/FLUX.1-dev'
LORA_REPO = 'whrw/Flux-Uncensored-V2'

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

load_started = time.perf_counter()
transformer_4bit = FluxTransformer2DModel.from_pretrained(
    BASE_MODEL,
    subfolder='transformer',
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)
print('Quantized transformer load:', round(time.perf_counter() - load_started, 1), 's')
if hasattr(transformer_4bit, 'get_memory_footprint'):
    print('Transformer parameter footprint GiB:', round(transformer_4bit.get_memory_footprint() / 1024**3, 2))

pipe4 = FluxPipeline.from_pretrained(
    BASE_MODEL,
    transformer=transformer_4bit,
    torch_dtype=torch.float16,
    device_map='auto',
)
print('Pipeline placement:', getattr(pipe4, 'hf_device_map', 'device map not exposed'))

lora_started = time.perf_counter()
pipe4.load_lora_weights(LORA_REPO)
print('LoRA load:', round(time.perf_counter() - lora_started, 1), 's')
print('4-bit FLUX pipeline and LoRA are ready.')


## 4. Generate with the exact reference settings

This uses the same prose prompt, seed, dimensions, steps, CFG, and maximum sequence length as the original reference notebook. The timer below measures generation only, not model loading.

In [ ]:
from IPython.display import display
import time, torch

prompt = (
    'A beautifully illustrated adult woman dressed as a gothic witch, wearing an elaborate '
    'crimson velvet dress with delicate black lace and an oversized pointed hat. She holds '
    'a glowing red apple in one hand. Dark gothic architecture in the background, rich '
    'cinematic lighting, soft atmospheric shadows, intricate fabric texture, expressive '
    'anime-inspired character design, carefully composed full illustration.'
)
seed = 123456789
generator = torch.Generator(device='cpu').manual_seed(seed)

torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
result4 = pipe4(
    prompt=prompt,
    width=1024,
    height=1024,
    num_inference_steps=28,
    guidance_scale=3.5,
    max_sequence_length=512,
    generator=generator,
)
image4 = result4.images[0]
elapsed4 = time.perf_counter() - started
image4.save('flux-dev-uncensored-4bit-reference.png')
print(f'4-bit generation time: {elapsed4:.1f} s ({elapsed4/60:.1f} min)')
print('Peak allocated VRAM GiB:', round(torch.cuda.max_memory_allocated() / 1024**3, 2))
print('Peak reserved VRAM GiB:', round(torch.cuda.max_memory_reserved() / 1024**3, 2))
print('Baseline time: 3262.4 s | speedup: {:.2f}x'.format(3262.4 / elapsed4))
display(image4)


## 5. Compare

Compare the new image against `flux-dev-uncensored-reference.png`. Same seed does not guarantee pixel-identical output after quantization. Look for changes in face, hands, fabric texture, fine details, and overall composition.

If this still takes an extremely long time, send the full output including the printed pipeline placement. Do not switch back to sequential CPU offload; the next experiment will be block-level group offloading or a different quantization backend.